# 01 — Data Inventory

Creates an inventory of all files under `data/raw` and summarizes file counts,
extensions and sizes.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd

records = []
for path in sorted(RAW_DIR.rglob("*")):
    if path.is_file():
        records.append({
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "folder": str(path.parent.relative_to(RAW_DIR)),
            "name": path.name,
            "extension": path.suffix.lower(),
            "size_mb": round(path.stat().st_size / (1024 ** 2), 4),
        })

inventory = pd.DataFrame(records)
display(inventory.head(20))
print(f"Total files: {len(inventory)}")

In [ ]:
summary = (
    inventory.groupby(["folder", "extension"], dropna=False)
    .agg(file_count=("name", "count"), total_size_mb=("size_mb", "sum"))
    .reset_index()
    .sort_values(["folder", "extension"])
)

output = INTERIM_DIR / "data_inventory.csv"
inventory.to_csv(output, index=False)
display(summary)
print(f"Saved: {output}")